<a href="https://colab.research.google.com/github/banyai-bence/adat-13c-02/blob/main/adattisztitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bolyai Labs – Feladatkártya #2: A hívásnapló megtisztítása

> Adatelemzőként szeretném, hogy a hívásnapló tiszta és egységes legyen, hogy a következő blokkokban a modellek megbízható adatból tanuljanak.

**Így dolgozz:** futtasd a cellákat sorban (Shift+Enter). Ahol `...  # TODO` áll, ott a `...` helyére írd a megoldást. A végén az **önellenőrző cella** megmondja, jó-e az eredmény.

**Mentés:** File → Save a copy in GitHub → a saját repód, `main` ág, a fájlnév maradjon `adattisztitas.ipynb`. Commitüzenet: `feat: hívásnapló tisztítása (feladatkártya #2)`.

## 1. Feltárás – mit rontottak el az adatban?

Futtasd az alábbi cellát, és nézd meg alaposan a kimenetet. Utána válaszolj a kérdésekre a következő szöveges cellában.

In [ ]:
from pathlib import Path
import pandas as pd

# Adatbetöltés
NYERS = Path("data/hivasok_nyers.csv")
SABLON_URL = "https://raw.githubusercontent.com/Kizsi2025/bolyai-labs-02-adat/main/data/hivasok_nyers.csv"

df = pd.read_csv(NYERS if NYERS.exists() else SABLON_URL, dtype=str)

print(df.shape)
display(df.head(10))

# 1. Feltárás

print("Hiányzó értékek:")
print(df.isna().sum())

print("Duplikált sorok:", df.duplicated().sum())

print("Modellnevek:")
print(df["modell"].value_counts())
print("Írásmódok száma:", df["modell"].nunique())

print("Sikeres oszlop értékei:")
print(df["sikeres"].value_counts())

print("Nem szám válaszidők:")
print(df.loc[pd.to_numeric(df["valaszido_ms"], errors="coerce").isna() & df["valaszido_ms"].notna(), "valaszido_ms"].value_counts())


# 2. Tisztítás

# a) Duplikált sorok törlése
df = df.drop_duplicates()
print(df.shape)


# b) Modellnevek egységesítése
df["modell"] = df["modell"].str.strip().str.lower()
print(df["modell"].value_counts())


# c) Sikeres oszlop átalakítása
ERTEKEK = {
    "1": 1, "igen": 1, "yes": 1, "true": 1,
    "0": 0, "nem": 0, "no": 0, "false": 0
}

df["sikeres"] = df["sikeres"].str.lower().map(ERTEKEK)
print(df["sikeres"].value_counts())


# d) Számmá alakítás
df["valaszido_ms"] = pd.to_numeric(df["valaszido_ms"], errors="coerce")
df["tokenek"] = pd.to_numeric(df["tokenek"])
print(df["valaszido_ms"].isna().sum())


# e) Kiugró értékek eltávolítása
ALSO, FELSO = 50, 10000

df = df[df["valaszido_ms"].isna() | df["valaszido_ms"].between(ALSO, FELSO)].copy()

print(df.shape)


# f) Hiányzó válaszidők pótlása
df["valaszido_ms"] = df["valaszido_ms"].fillna(
    df.groupby("modell")["valaszido_ms"].transform("median")
)

print(df["valaszido_ms"].isna().sum())


# g) Min-max normalizálás
v = df["valaszido_ms"]

df["valaszido_norm"] = (v - v.min()) / (v.max() - v.min())

print(df["valaszido_norm"].describe())


# 3. Önellenőrzés

assert not df.duplicated().any(), "Maradt duplikált sor."

assert set(df["modell"]) == {"kisebb", "nagyobb"}, "Hibás modellnevek."

assert set(df["sikeres"]) <= {0, 1}, "Hibás sikeres oszlop."

assert df["valaszido_ms"].between(50, 10000).all(), "Hibás válaszidő."

assert abs(df["valaszido_norm"].min()) < 1e-9
assert abs(df["valaszido_norm"].max() - 1) < 1e-9

assert len(df) == 214, "Hibás sorszám."
assert round(df["valaszido_ms"].mean(), 1) == 688.6

print(f"Minden ellenőrzés rendben! {len(df)} tiszta sor.")


# 4. Mentés

Path("data").mkdir(exist_ok=True)

df.to_csv("data/hivasok_tiszta.csv", index=False)

print("Mentve: data/hivasok_tiszta.csv")

(232, 6)


,hivas_id,datum,modell,valaszido_ms,tokenek,sikeres
0,94,2026-10-12,nagyobb,911,980,no
1,4,2026-10-01,nagyobb,981,740,True
2,181,2026-10-23,nagyobb,914,787,1
3,130,2026-10-17,kisebb,355,459,yes
4,60,2026-10-08,kisebb,435,420,igen
5,87,2026-10-11,nagyobb,1074,1056,1
6,7,2026-10-01,nagyobb,1219,449,igen
7,177,2026-10-23,kisebb,hiba,432,1
8,188,2026-10-24,kisebb,348,382,1
9,198,2026-10-25,nagyobb,941,875,yes


Hiányzó értékek:
hivas_id         0
datum            0
modell           0
valaszido_ms    13
tokenek          0
sikeres          0
dtype: int64
Duplikált sorok: 12
Modellnevek:
modell
nagyobb     111
kisebb       90
nagyobb       7
 NAGYOBB      6
kisebb        5
Nagyobb       5
 Kisebb       4
KISEBB        4
Name: count, dtype: int64
Írásmódok száma: 8
Sikeres oszlop értékei:
sikeres
1        104
igen      43
yes       35
True      33
0          7
no         4
nem        4
False      2
Name: count, dtype: int64
Nem szám válaszidők:
valaszido_ms
hiba    3
Name: count, dtype: int64
(220, 6)
modell
nagyobb    123
kisebb      97
Name: count, dtype: int64
sikeres
1    204
0     16
Name: count, dtype: int64
14
(214, 6)
0
count    214.000000
mean       0.396952
std        0.284970
min        0.000000
25%        0.104735
50%        0.428571
75%        0.641653
max        1.000000
Name: valaszido_norm, dtype: float64
Minden ellenőrzés rendben! 214 tiszta sor.
Mentve: data/hivasok_tiszta.csv


**Válaszolj ide (dupla kattintás a cellára):**

1. Hány duplikált sor van az adatban?
2. Hányféleképpen írták a modell nevét?
3. Milyen nem szám értékek vannak a válaszidő oszlopban?
4. Szerinted reális-e a 999 999 ms-os válaszidő? Miért?

## 2. Tisztítás – hét lépés, ebben a sorrendben

A sorrend számít: ha felcseréled, más eredményt kapsz, és az önellenőrzés jelezni fogja.

## 3. Önellenőrzés

Ha ez a cella hiba nélkül lefut, a tisztítás jó. Ha hibát ír, az üzenet megmondja, melyik lépést nézd át.

## 4. Mentés

Ez a cella elmenti a tiszta adatot. A CI ugyanezt a notebookot futtatja le, és a mentett fájlt ellenőrzi.